### UK Access, Watch, Reserve, and Other classification for antibiotics in dmd

The ["UK Access, Watch, Reserve, and Other classification for antibiotics"](https://www.gov.uk/government/publications/uk-aware-antibiotic-classification/uk-access-watch-reserve-and-other-classification-for-antibiotics-uk-aware-antibiotic-classification) categorises antibiotics into groups such as Access, Watch, and Reserve to guide healthcare professionals in optimising their use and mitigating antimicrobial resistance.

The existing list on the UKHSA site is not machine readable. To facilitate research, it is helpful to link this categorisation to the [dm+d standard](https://www.nhsbsa.nhs.uk/pharmacies-gp-practices-and-appliance-contractors/dictionary-medicines-and-devices-dmd).

### Imports
We need to import some libaries to help with the code

In [13]:
import pandas as pd
import requests
from ebmdatalab import bq
import os

### Getting the AWaRe list
The AWaRe list is currently available on the following webpage. We can read the html page directly into Pandas to retrieve the first table.

In [14]:
# Set URL to scrape
url = "https://www.gov.uk/government/publications/uk-aware-antibiotic-classification/uk-access-watch-reserve-and-other-classification-for-antibiotics-uk-aware-antibiotic-classification"

# Read the first HTML table from the page into a DataFrame using the HTML content
df_scrape = pd.read_html(url)[0]

# Display the DataFrame
df_scrape

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category
0,Amikacin,Watch,Watch
1,Amoxicillin,Access,Access
2,Amoxicillin/ clavulanic-acid,Watch,Watch
3,Ampicillin,Access,Access
4,Azithromycin,Watch,Watch
...,...,...,...
85,Tigecycline,Reserve,Reserve
86,Tinidazole #,Other,Access
87,Tobramycin,Watch,Watch
88,Trimethoprim,Access,Access


The [WHO produce an AWaRe list](https://www.who.int/publications/i/item/WHO-MHP-HPS-EML-2023.04) from which the UK version is adapted. In the WHO version ATC codes are mapped to Antibiotic name.


In [15]:
who_to_atc_path = os.path.join('..', 'data', 'WHO_Antibiotic_to_ATC.csv')
who_antibiotic_to_atc = pd.read_csv(who_to_atc_path)
who_antibiotic_to_atc

,Antibiotic,Class,ATC code
0,Amikacin,Aminoglycosides,J01GB06
1,Amoxicillin,Penicillins,J01CA04
2,Amoxicillin/clavulanic-acid,Beta-lactam/beta-lactamase-inhibitor,J01CR02
3,Ampicillin,Penicillins,J01CA01
4,Ampicillin/sulbactam,Beta-lactam/beta-lactamase-inhibitor,J01CR01
...,...,...,...
252,Trimethoprim,Trimethoprim-derivatives,J01EA01
253,Troleandomycin,Macrolides,J01FA08
254,Trovafloxacin,Fluoroquinolones,J01MA13
255,Vancomycin_IV,Glycopeptides,J01XA01


### Cleaning up the UKHSA list
The list contains some characters or identifiers that could break matching, we need to clean these up.

The # symbol is used to denote where category has changed, remove this

In [16]:
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace(" #", "", regex=False)

We can review remaining rows that contain non-alphabetical characters

In [17]:
df_scrape[df_scrape['Antibiotic'].str.contains(r'[^A-Za-z ]', na=False)]

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category
2,Amoxicillin/ clavulanic-acid,Watch,Watch
6,Benzathine-benzylpenicillin,Access,Access
20,Ceftaroline-fosamil,Reserve,Reserve
22,Ceftazidime/ avibactam,Reserve,Reserve
23,Ceftobiprole-medocaril,Reserve,Reserve
24,Ceftolozane/ tazobactam,Reserve,Reserve
31,"Colistin, intravenous",Reserve,Reserve
32,"Colistin, oral",Reserve,Reserve
34,Dalfopristin/ quinupristin,Reserve,Reserve
46,"Fosfomycin, intravenous",Reserve,Reserve


Some rows specify a route. Sometimes there is a comma between antibiotic and route, sometimes there isn't. We can pull any specified route into a seperate column.

In [18]:
# List of routes to check.
route_list = ["oral", "intravenous"]

# Function to extract the route if the Antibiotic entry ends with a route.
def extract_route(antibiotic):
    for route in route_list:
        if antibiotic.endswith(" " + route):
            return route
    return ""  # Return empty string if no route is found.

# Apply the function to create a new 'Route' column.
df_scrape['Route_specific'] = df_scrape['Antibiotic'].apply(extract_route)

# Remove any route names from the 'Antibiotic' column only if they appear at the end
# and are preceded by either ", " or " ".
for route in route_list:
    df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace(fr'(, | ){route}$', '', regex=True)

# Define mapping
route_map = {'oral': 'O', 'intravenous': 'P'}

# Replace NaN with empty strings just to be safe
df_scrape['Route_specific'] = df_scrape['Route_specific'].fillna('')

# Map values using a dictionary, and fill any unmapped entries with empty string
df_scrape['ATC_Route_specific'] = df_scrape['Route_specific'].map(route_map).fillna('')

df_scrape

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category,Route_specific,ATC_Route_specific
0,Amikacin,Watch,Watch,,
1,Amoxicillin,Access,Access,,
2,Amoxicillin/ clavulanic-acid,Watch,Watch,,
3,Ampicillin,Access,Access,,
4,Azithromycin,Watch,Watch,,
...,...,...,...,...,...
85,Tigecycline,Reserve,Reserve,,
86,Tinidazole,Other,Access,,
87,Tobramycin,Watch,Watch,,
88,Trimethoprim,Access,Access,,


In [19]:
df_scrape[df_scrape['Antibiotic'].str.contains(r'[^A-Za-z ]', na=False)]

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category,Route_specific,ATC_Route_specific
2,Amoxicillin/ clavulanic-acid,Watch,Watch,,
6,Benzathine-benzylpenicillin,Access,Access,,
20,Ceftaroline-fosamil,Reserve,Reserve,,
22,Ceftazidime/ avibactam,Reserve,Reserve,,
23,Ceftobiprole-medocaril,Reserve,Reserve,,
24,Ceftolozane/ tazobactam,Reserve,Reserve,,
34,Dalfopristin/ quinupristin,Reserve,Reserve,,
49,Imipenem/cilastatin,Reserve,Reserve,,
50,Imipenem/cilastatin/relebactam,Reserve,Reserve,,
55,Meropenem/ vaborbactam,Reserve,Reserve,,


Some of the formatting in the UKHSA is slightly inconsistent with the WHO format.
In WHO format there is no space after / for combo products. Remove these from UKHSA version.
In WHO format spaces between parts of antibiotic name are always a dash (-) replace spaces for dashes.

In [20]:
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace('/ ', '/', regex=False)
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace(' ', '-', regex=False)

In [23]:
with pd.option_context('display.max_rows', None):
    display(df_scrape)

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category,Route_specific,ATC_Route_specific
0,Amikacin,Watch,Watch,,
1,Amoxicillin,Access,Access,,
2,Amoxicillin/clavulanic-acid,Watch,Watch,,
3,Ampicillin,Access,Access,,
4,Azithromycin,Watch,Watch,,
5,Aztreonam,Reserve,Reserve,,
6,Benzathine-benzylpenicillin,Access,Access,,
7,Benzylpenicillin,Access,Access,,
8,Cefaclor,Watch,Watch,,
9,Cefadroxil,Watch,Access,,


## Cleaning up the WHO to ATC list

The WHO list also contains some routes within the names. Seperate this out.

In [21]:
# List of routes to check.
route_list = ["oral", "IV"]

# Function to extract the route if the Antibiotic entry ends with a route.
def extract_route(antibiotic):
    for route in route_list:
        if antibiotic.endswith("_" + route):
            return route
    return ""  # Return empty string if no route is found.

# Apply the function to create a new 'Route' column.
who_antibiotic_to_atc['Route_specific'] = who_antibiotic_to_atc['Antibiotic'].apply(extract_route)

# Remove any route names from the 'Antibiotic' column only if they appear at the end
# and are preceded by either ", " or " ".
for route in route_list:
    who_antibiotic_to_atc['Antibiotic'] = who_antibiotic_to_atc['Antibiotic'].str.replace(fr'(_| ){route}$', '', regex=True)

# Define mapping
route_map = {'oral': 'O', 'IV': 'P'}

# Replace NaN with empty strings just to be safe
who_antibiotic_to_atc['Route_specific'] = who_antibiotic_to_atc['Route_specific'].fillna('')

# Map values using a dictionary, and fill any unmapped entries with empty string
who_antibiotic_to_atc['ATC_Route_specific'] = who_antibiotic_to_atc['Route_specific'].map(route_map).fillna('')

,Antibiotic,Class,ATC code,Route_specific,ATC_Route_specific
0,Amikacin,Aminoglycosides,J01GB06,,
1,Amoxicillin,Penicillins,J01CA04,,
2,Amoxicillin/clavulanic-acid,Beta-lactam/beta-lactamase-inhibitor,J01CR02,,
3,Ampicillin,Penicillins,J01CA01,,
4,Ampicillin/sulbactam,Beta-lactam/beta-lactamase-inhibitor,J01CR01,,
...,...,...,...,...,...
252,Trimethoprim,Trimethoprim-derivatives,J01EA01,,
253,Troleandomycin,Macrolides,J01FA08,,
254,Trovafloxacin,Fluoroquinolones,J01MA13,,
255,Vancomycin,Glycopeptides,J01XA01,IV,P


In [24]:
with pd.option_context('display.max_rows', None):
    display(who_antibiotic_to_atc)

,Antibiotic,Class,ATC code,Route_specific,ATC_Route_specific
0,Amikacin,Aminoglycosides,J01GB06,,
1,Amoxicillin,Penicillins,J01CA04,,
2,Amoxicillin/clavulanic-acid,Beta-lactam/beta-lactamase-inhibitor,J01CR02,,
3,Ampicillin,Penicillins,J01CA01,,
4,Ampicillin/sulbactam,Beta-lactam/beta-lactamase-inhibitor,J01CR01,,
5,Arbekacin,Aminoglycosides,J01GB12,,
6,Aspoxicillin,Penicillins,J01CA19,,
7,Azidocillin,Penicillins,J01CE04,,
8,Azithromycin,Macrolides,J01FA10,,
9,Azlocillin,Penicillins,J01CA09,,


## Adding ATC codes to UKHSA list

We can merge the ATC codes from WHO file with the Antibiotic list published by the UKHSA.

In [25]:
ukhsa_list_with_atc = df_scrape.merge(
    who_antibiotic_to_atc,
    how='left',
    left_on=['Antibiotic', 'ATC_Route_specific'],
    right_on=['Antibiotic', 'ATC_Route_specific']
)

with pd.option_context('display.max_rows', None):
    display(ukhsa_list_with_atc)

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category,Route_specific_x,ATC_Route_specific,Class,ATC code,Route_specific_y
0,Amikacin,Watch,Watch,,,Aminoglycosides,J01GB06,
1,Amoxicillin,Access,Access,,,Penicillins,J01CA04,
2,Amoxicillin/clavulanic-acid,Watch,Watch,,,Beta-lactam/beta-lactamase-inhibitor,J01CR02,
3,Ampicillin,Access,Access,,,Penicillins,J01CA01,
4,Azithromycin,Watch,Watch,,,Macrolides,J01FA10,
5,Aztreonam,Reserve,Reserve,,,Monobactams,J01DF01,
6,Benzathine-benzylpenicillin,Access,Access,,,Penicillins,J01CE08,
7,Benzylpenicillin,Access,Access,,,Penicillins,J01CE01,
8,Cefaclor,Watch,Watch,,,Second-generation-cephalosporins,J01DC04,
9,Cefadroxil,Watch,Access,,,First-generation-cephalosporins,J01DB05,


Some are missing matches - where there is a single obvious ATC code match we can add in the appropriate code.

In [27]:
ukhsa_list_with_atc.loc[ukhsa_list_with_atc['Antibiotic'] == 'Methenamine', ['Class', 'ATC code']] = ['Triazinanes', 'J01XX05']
ukhsa_list_with_atc.loc[ukhsa_list_with_atc['Antibiotic'] == 'Nalidixic-Acid', ['Class', 'ATC code']] = ['Quinolone', 'J01MB02']
ukhsa_list_with_atc.loc[ukhsa_list_with_atc['Antibiotic'] == 'Sulfadiazine', ['Class', 'ATC code']] = ['Sulfonamides', 'J01EC02']


Some match to multiple ATC codes (different for oral and IV) - we can add in new rows for this and remove the unmatched rows.

In [29]:
manual_entries = [
    {
        "Antibiotic": "Metronidazole",
        "England-adapted 2019 AWaRe category": "Access",
        "UK-adapted 2024 AWaRe category": "Access",
        "Route_specific_x": "oral",
        "ATC_Route_specific": "O",
        "Class": "Imidazoles",
        "ATC code": "P01AB01",
        "Route_specific_y": "oral"
    },
    {
        "Antibiotic": "Metronidazole",
        "England-adapted 2019 AWaRe category": "Access",
        "UK-adapted 2024 AWaRe category": "Access",
        "Route_specific_x": "intravenous",
        "ATC_Route_specific": "P",
        "Class": "Imidazoles",
        "ATC code": "J01XD01",
        "Route_specific_y": "IV"
    },
    {
        "Antibiotic": "Tinidazole",
        "England-adapted 2019 AWaRe category": "Other",
        "UK-adapted 2024 AWaRe category": "Access",
        "Route_specific_x": "oral",
        "ATC_Route_specific": "O",
        "Class": "Imidazoles",
        "ATC code": "P01AB02",
        "Route_specific_y": "oral"
    },
    {
        "Antibiotic": "Tinidazole",
        "England-adapted 2019 AWaRe category": "Other",
        "UK-adapted 2024 AWaRe category": "Access",
        "Route_specific_x": "intravenous",
        "ATC_Route_specific": "P",
        "Class": "Imidazoles",
        "ATC code": "J01XD02",
        "Route_specific_y": "IV"
    },
    {
        "Antibiotic": "Vancomycin",
        "England-adapted 2019 AWaRe category": "Watch",
        "UK-adapted 2024 AWaRe category": "Watch",
        "Route_specific_x": "oral",
        "ATC_Route_specific": "O",
        "Class": "Imidazoles",
        "ATC code": "A07AA09",
        "Route_specific_y": "oral"
    },
    {
        "Antibiotic": "Vancomycin",
        "England-adapted 2019 AWaRe category": "Watch",
        "UK-adapted 2024 AWaRe category": "Watch",
        "Route_specific_x": "intravenous",
        "ATC_Route_specific": "P",
        "Class": "Imidazoles",
        "ATC code": "J01XA01",
        "Route_specific_y": "IV"
    },
    {
        "Antibiotic": "Neomycin",
        "England-adapted 2019 AWaRe category": "Watch",
        "UK-adapted 2024 AWaRe category": "Watch",
        "Route_specific_x": "oral",
        "ATC_Route_specific": "O",
        "Class": "Imidazoles",
        "ATC code": "A07AA01",
        "Route_specific_y": "oral"
    },
    {
        "Antibiotic": "Neomycin",
        "England-adapted 2019 AWaRe category": "Watch",
        "UK-adapted 2024 AWaRe category": "Watch",
        "Route_specific_x": "intravenous",
        "ATC_Route_specific": "P",
        "Class": "Imidazoles",
        "ATC code": "J01GB05",
        "Route_specific_y": "IV"
    },
]

In [30]:
manual_df = pd.DataFrame(manual_entries)

# Remove existing rows where Antibiotic is one of those in the manual entries
ukhsa_cleaned = ukhsa_list_with_atc[~ukhsa_list_with_atc['Antibiotic'].isin(manual_df['Antibiotic'])]

# Append the cleaned manual rows
ukhsa_final = pd.concat([ukhsa_cleaned, manual_df], ignore_index=True)

Reorder the columns and rename to simplify

In [34]:
ukhsa_final = ukhsa_final[['Antibiotic', 'Route_specific_x', 'Class', 'ATC code', 'England-adapted 2019 AWaRe category', 'UK-adapted 2024 AWaRe category']]

ukhsa_final.rename(columns={'England-adapted 2019 AWaRe category': 'aware_2019'}, inplace=True)
ukhsa_final.rename(columns={'UK-adapted 2024 AWaRe category': 'aware_2024'}, inplace=True)
ukhsa_final.rename(columns={'Route_specific_x': 'route'}, inplace=True)
ukhsa_final.rename(columns={'ATC code': 'ATC code'}, inplace=True)

with pd.option_context('display.max_rows', None):
    display(ukhsa_final)

,Antibiotic,route,Class,ATC code,aware_2019,aware_2024
0,Amikacin,,Aminoglycosides,J01GB06,Watch,Watch
1,Amoxicillin,,Penicillins,J01CA04,Access,Access
2,Amoxicillin/clavulanic-acid,,Beta-lactam/beta-lactamase-inhibitor,J01CR02,Watch,Watch
3,Ampicillin,,Penicillins,J01CA01,Access,Access
4,Azithromycin,,Macrolides,J01FA10,Watch,Watch
5,Aztreonam,,Monobactams,J01DF01,Reserve,Reserve
6,Benzathine-benzylpenicillin,,Penicillins,J01CE08,Access,Access
7,Benzylpenicillin,,Penicillins,J01CE01,Access,Access
8,Cefaclor,,Second-generation-cephalosporins,J01DC04,Watch,Watch
9,Cefadroxil,,First-generation-cephalosporins,J01DB05,Watch,Access


In [35]:
ukhsa_final.to_csv("ukhsa_atc.csv", index=False)

## Matching VMPs

We can attempt to match Antibiotic names in the UKHSA to VTMs in dm+d